In [48]:
print(123)

123


In [49]:
import pandas as pd
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet"


In [50]:
columns = ['PULocationID', 'DOLocationID', 'trip_distance', 'total_amount', 'tpep_pickup_datetime']
df = pd.read_parquet(url, columns=columns).head(1000)


In [51]:
df.head()


,PULocationID,DOLocationID,trip_distance,total_amount,tpep_pickup_datetime
0,43,186,1.68,22.15,2025-11-01 00:13:25
1,142,237,2.28,24.94,2025-11-01 00:49:07
2,163,238,2.70,25.62,2025-11-01 00:07:19
3,138,261,12.87,86.14,2025-11-01 00:00:00
4,138,37,8.40,48.65,2025-11-01 00:18:50


In [52]:
from dataclasses import dataclass

@dataclass
class TaxiRide:
    PULocationID: int
    DOLocationID: int
    trip_distance: float
    total_amount: float
    tpep_pickup_datetime: int




In [53]:
def ride_from_row(row):
    return TaxiRide(
        PULocationID=int(row['PULocationID']),
        DOLocationID=int(row['DOLocationID']),
        trip_distance=float(row['trip_distance']),
        total_amount=float(row['total_amount']),
        tpep_pickup_datetime=int(row['tpep_pickup_datetime'].timestamp() * 1000),
    )

In [54]:
ride = ride_from_row(df.iloc[0])
ride


TaxiRide(PULocationID=43, DOLocationID=186, trip_distance=1.68, total_amount=22.15, tpep_pickup_datetime=1761956005000)

In [55]:
import json
import dataclasses

def ride_serializer(ride):
    # Expects a TaxiRide dataclass instance (not a dict)
    ride_dict = dataclasses.asdict(ride)
    return json.dumps(ride_dict).encode("utf-8")

In [56]:
import json
from kafka import KafkaProducer

def json_serializer(data):
    return json.dumps(data).encode('utf-8')

In [57]:
server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)


In [58]:
topic_name = "rides"

# Pass the TaxiRide object — ride_serializer converts it to JSON bytes
producer.send(topic_name, value=ride)
producer.flush()
print("sent:", ride)

sent: TaxiRide(PULocationID=43, DOLocationID=186, trip_distance=1.68, total_amount=22.15, tpep_pickup_datetime=1761956005000)


In [60]:
import time

t0 = time.time()
topic_name = "rides"

for i, row in df.iterrows():
    ride_msg = ride_from_row(row)
    producer.send(topic_name, value=ride_msg)
    if i % 100 == 0:
        print(f"Sent row {i}: {ride_msg}")

producer.flush()
t1 = time.time()
print(f"Sent {len(df)} rides in {(t1 - t0):.2f} seconds")

Sent row 0: TaxiRide(PULocationID=43, DOLocationID=186, trip_distance=1.68, total_amount=22.15, tpep_pickup_datetime=1761956005000)
Sent row 100: TaxiRide(PULocationID=144, DOLocationID=186, trip_distance=2.9, total_amount=24.85, tpep_pickup_datetime=1761958647000)
Sent row 200: TaxiRide(PULocationID=161, DOLocationID=7, trip_distance=4.12, total_amount=31.8, tpep_pickup_datetime=1761955283000)
Sent row 300: TaxiRide(PULocationID=79, DOLocationID=68, trip_distance=2.16, total_amount=28.95, tpep_pickup_datetime=1761956224000)
Sent row 400: TaxiRide(PULocationID=170, DOLocationID=142, trip_distance=2.1, total_amount=22.25, tpep_pickup_datetime=1761957923000)
Sent row 500: TaxiRide(PULocationID=148, DOLocationID=87, trip_distance=3.04, total_amount=26.05, tpep_pickup_datetime=1761958064000)
Sent row 600: TaxiRide(PULocationID=138, DOLocationID=162, trip_distance=9.95, total_amount=76.61, tpep_pickup_datetime=1761956513000)
Sent row 700: TaxiRide(PULocationID=144, DOLocationID=141, trip_di